In [6]:
import csv
import math
from pathlib import Path

DATA_PATH = Path("data") / "agent_deposits.csv"


def load_column(path: Path, column: str) -> tuple[list[float], list[tuple[int, str]]]:
    valid_floats = []
    errors = []

    with open(path, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)

        for row_idx, row in enumerate(reader, start=2):
            raw_value = row.get(column)

            try:
                
                value = float(raw_value)
                valid_floats.append(value)

            except ValueError:
                errors.append((row_idx, raw_value))

    # return f"Valid values {valid_floats}", f"Invalid values {errors}"
    return valid_floats, errors


load_column(DATA_PATH, "transactions")

([100.0,
  89.0,
  125.0,
  96.0,
  63.0,
  68.0,
  132.0,
  73.0,
  111.0,
  73.0,
  97.0,
  109.0,
  62.0,
  60.0,
  87.0,
  86.0,
  66.0,
  120.0,
  108.0,
  110.0,
  113.0,
  69.0,
  132.0,
  140.0,
  85.0,
  94.0,
  103.0,
  71.0,
  99.0,
  102.0,
  61.0,
  112.0,
  75.0,
  77.0,
  91.0],
 [(7, ''), (10, ''), (14, ''), (17, 'abc'), (29, '')])

In [8]:

def mean(values: list[float]) -> float:
    if not values:
        raise ValueError("Cannot compute mean of an empty list.")
    return sum (values) / len(values)
print(f"{mean(load_column(DATA_PATH, "transactions")[0]):.2f}")


def median(values: list[float]) -> float:
    if not values:
        raise ValueError("Cannot compute median of an empty list.")
    sorted_values = sorted(values)
    n = len(sorted_values)
    mid = n // 2
    if n % 2 == 0:
        return (sorted_values[mid - 1] + sorted_values[mid]) / 2
    else:
        return sorted_values[mid]
print(median(load_column(DATA_PATH, "transactions")[0]))
     

def mode(values: list[float]) -> list[float]:
    """Return every value that occurs most often (a list, because there can be ties or none)."""
    if not values:
        raise ValueError("Cannot compute mode of an empty list.")
    frequency = {}
    for value in values:
        frequency[value] = frequency.get(value, 0) + 1
    max_freq = max(frequency.values())
    modes = [value for value, freq in frequency.items() if freq == max_freq]
    return modes
print(mode(load_column(DATA_PATH, "transactions")[0]))

    

93.11
94.0
[132.0, 73.0]


In [9]:
def variance(values: list[float], sample: bool = True) -> float:
    """Sample variance (n - 1) by default; population variance (n) when sample is False."""
    if not values:
        raise ValueError("Cannot compute variance of an empty list.")
    n = len(values)
    if sample and n < 2:
        raise ValueError("Sample variance requires at least two data points.")
    mean_value = mean(values)
    sum_squared_diff = sum((x - mean_value) ** 2 for x in values)
    return sum_squared_diff / (n - 1) if sample else sum_squared_diff / n
print(f"{variance(load_column(DATA_PATH, "transactions")[0]):.2f}")

502.57


In [10]:
def std_dev(values: list[float], sample: bool = True) -> float:
    """Sample standard deviation (n - 1) by default; population standard deviation (n) when sample is False."""
    return math.sqrt(variance(values, sample))
print(f"{std_dev(load_column(DATA_PATH, "transactions")[0]):.2f}")

22.42


In [11]:

def quartiles(values: list[float]) -> tuple[float, float, float]:
    """Return (Q1, Q2, Q3). Document which quartile method you chose."""
    if not values:
        raise ValueError("Cannot compute quartiles of an empty list.")
    sorted_values = sorted(values)
    n = len(sorted_values)
    q1_idx = n // 4
    q2_idx = n // 2
    q3_idx = 3 * n // 4
    return (sorted_values[q1_idx], sorted_values[q2_idx], sorted_values[q3_idx])
print(f"Quartiles: {quartiles(load_column(DATA_PATH, "transactions")[0])}")

Quartiles: (73.0, 94.0, 110.0)


In [12]:
def iqr_outliers(values: list[float], k: float = 1.5) -> dict:
    """Return the IQR, the lower and upper fences, and the values outside them."""
    if not values:
        raise ValueError("Cannot compute IQR outliers of an empty list.")
    q1, q2, q3 = quartiles(values)
    iqr = q3 - q1
    lower_fence = q1 - k * iqr
    upper_fence = q3 + k * iqr
    outliers = [x for x in values if x < lower_fence or x > upper_fence]
    return {
        "IQR": iqr,
        "Lower Fence": lower_fence,
        "Upper Fence": upper_fence,
        "Outliers": outliers
    }
print(f"IQR Outliers: {iqr_outliers(load_column(DATA_PATH, "transactions")[0])}")

IQR Outliers: {'IQR': 37.0, 'Lower Fence': 17.5, 'Upper Fence': 165.5, 'Outliers': []}


In [19]:
class StatisticsAnalyzer:
    """Wraps one numeric column and exposes every statistic above as a method."""

    def __init__(self, values: list[float], name: str = "values") -> None:
        self.values = values
        self.name = name
        print(f"StatisticsAnalyzer created for column '{self.name}' with {len(self.values)} valid values.")

    @classmethod
    def from_csv(cls, path: Path, column: str) -> "StatisticsAnalyzer":
        values, errors = load_column(path, column)
        if errors:
            print(f"Warning: Found {len(errors)} invalid values in column '{column}': {errors}")
        return cls(values, name=column)
    

    def summary(self) -> dict:
        if not self.values:
            raise ValueError("Cannot generate summary for an empty list of values.")
        return {
            "mean": mean(self.values),
            "median": median(self.values),
            "mode": mode(self.values),
            "variance": variance(self.values),
            "std_dev": std_dev(self.values),
            "quartiles": quartiles(self.values),
            "iqr_outliers": iqr_outliers(self.values),
        }
    

    def without_outliers(self) -> "StatisticsAnalyzer":
        outlier_info = iqr_outliers(self.values)
        filtered_values = [x for x in self.values if x not in outlier_info["Outliers"]]
        return StatisticsAnalyzer(filtered_values, name=f"{self.name} (without outliers)")
    

    def report(self) -> str:
        if not self.values:
            raise ValueError("Cannot generate report for an empty list of values.")
        summary = self.summary()
        report = f"Report for column '{self.name}':\n"
        for stat, value in summary.items():
            report += f"  {stat.capitalize()}: {value}\n"
        return report

    from pathlib import Path

# 1. Build the analyzer from your CSV (change the column name to match your file)
analyzer = StatisticsAnalyzer.from_csv(DATA_PATH, "transactions")

# 2. Full report (formatted string)
print(analyzer.report())

# 3. Or the raw summary dict
print(analyzer.summary())

# 4. Repeat the report with outliers removed
clean = analyzer.without_outliers()
print(clean.report())

StatisticsAnalyzer created for column 'transactions' with 35 valid values.
Report for column 'transactions':
  Mean: 93.11428571428571
  Median: 94.0
  Mode: [132.0, 73.0]
  Variance: 502.5747899159664
  Std_dev: 22.418179897484237
  Quartiles: (73.0, 94.0, 110.0)
  Iqr_outliers: {'IQR': 37.0, 'Lower Fence': 17.5, 'Upper Fence': 165.5, 'Outliers': []}

{'mean': 93.11428571428571, 'median': 94.0, 'mode': [132.0, 73.0], 'variance': 502.5747899159664, 'std_dev': 22.418179897484237, 'quartiles': (73.0, 94.0, 110.0), 'iqr_outliers': {'IQR': 37.0, 'Lower Fence': 17.5, 'Upper Fence': 165.5, 'Outliers': []}}
StatisticsAnalyzer created for column 'transactions (without outliers)' with 35 valid values.
Report for column 'transactions (without outliers)':
  Mean: 93.11428571428571
  Median: 94.0
  Mode: [132.0, 73.0]
  Variance: 502.5747899159664
  Std_dev: 22.418179897484237
  Quartiles: (73.0, 94.0, 110.0)
  Iqr_outliers: {'IQR': 37.0, 'Lower Fence': 17.5, 'Upper Fence': 165.5, 'Outliers': []}


In [21]:
def compare_with_numpy(analyzer: StatisticsAnalyzer) -> dict[str, bool]:
    """Import numpy HERE, recompute each statistic and return {name: matches}."""
    import numpy as np

    values = np.array(analyzer.values)
    summary = analyzer.summary()

    # NumPy mode: every value that appears the maximum number of times
    unique_vals, counts = np.unique(values, return_counts=True)
    numpy_modes = set(unique_vals[counts == counts.max()].tolist())

    results = {
        "mean": bool(np.isclose(summary["mean"], np.mean(values))),
        "median": bool(np.isclose(summary["median"], np.median(values))),
        "mode": set(summary["mode"]) == numpy_modes,
        "variance": bool(np.isclose(summary["variance"], np.var(values, ddof=1))),
        "std_dev": bool(np.isclose(summary["std_dev"], np.std(values, ddof=1))),
    }
    return results


print(compare_with_numpy(analyzer))

{'mean': True, 'median': True, 'mode': True, 'variance': True, 'std_dev': True}
